# Clase REBOUND — Aproximación de Apophis a la Tierra (2029)

**Curso:** Mecánica Celeste  
**Tema:** Simulación de encuentros cercanos — El caso 99942 Apophis  
**Fecha:** 2026  

---

## Objetivos

1. Simular la aproximación del asteroide 99942 Apophis a la Tierra en 2029 usando REBOUND.
2. Establecer con precisión el **sistema de referencia** utilizado.
3. Obtener condiciones iniciales desde NASA Horizons y mediante elementos orbitales.
4. Analizar la geometría del encuentro cercano con múltiples gráficos.
5. Calcular la distancia mínima, tiempo de encuentro y velocidad relativa.

---

## Contexto histórico y científico

El asteroide **99942 Apophis** (inicialmente designado 2004 MN₄) fue descubierto el 19 de junio de 2004 por Roy Tucker, David Tholen y Fabrizio Bernardi en el Observatorio Nacional de Kitt Peak. Cuando se calcularon sus primeras órbitas, generó una preocupación considerable: en un primer momento se estimó una probabilidad de impacto con la Tierra del ~2.7% para el año 2029, asignándole un nivel 4 en la escala de Torino — el valor más alto jamás asignado a un asteroide observado.

Observaciones adicionales descartaron el impacto en 2029, pero confirmaron que **el 13 de abril de 2029** Apophis pasará a aproximadamente **31,600 km** de la superficie terrestre (distancia desde el centro de la Tierra: ~38,017 km ≈ 0.25 distancias lunares), lo que lo convertirá en el **objeto celeste natural más grande en acercarse tanto a la Tierra** en la historia registrada.

Esta aproximación es también de gran interés científico porque las fuerzas de marea y los efectos no gravitacionales (efecto Yarkovsky) durante el encuentro podrían modificar la órbita de Apophis de forma apreciable.

**Referencias principales:**
- Farnocchia, D. et al. (2021). *Apophis: The 2029 Earth Encounter*. The Planetary Science Journal, 2, 2.
- NASA JPL CNEOS: https://cneos.jpl.nasa.gov/ca/
- Brozović, M. et al. (2018). *Goldstone and Arecibo radar observations of (99942) Apophis in 2012–2013*. Icarus, 300, 115–128.

---

## 1. Sistema de referencia

### 1.1 Marco heliocéntrico eclíptico J2000.0

Todas las simulaciones de este cuaderno usan el **marco heliocéntrico eclíptico J2000.0**, que es el estándar en la mecánica del sistema solar:

| Eje | Dirección |
|-----|-----------|
| **x̂** | Hacia el equinoccio vernal (punto Aries ♈) en la época J2000.0 |
| **ŷ** | En el plano de la eclíptica, 90° delante de x̂ (sentido antihorario visto desde el norte) |
| **ẑ** | Normal al plano de la eclíptica, hacia el polo norte eclíptico |

**Origen:** Centro de masa del Sol (y efectivamente del sistema solar para integraciones cortas).

REBOUND mueve automáticamente el origen al **centro de masa del sistema** (`sim.move_to_com()`), por lo que técnicamente el marco es **baricéntrico eclíptico J2000.0**. Para un sistema Sol + Tierra + Apophis, la diferencia es despreciable.

### 1.2 Unidades del sistema REBOUND (unidades de N-cuerpos)

| Cantidad | Unidad | Equivalencia SI |
|----------|--------|-----------------|
| Longitud | 1 UA (Unidad Astronómica) | 1.496 × 10¹¹ m |
| Tiempo | 1 año = 2π u.t. | 3.156 × 10⁷ s |
| Masa | Masa del Sol M☉ | 1.989 × 10³⁰ kg |
| G | 4π² [UA³/(M☉·año²)] | 6.674 × 10⁻¹¹ m³/(kg·s²) |

Con estas unidades, la Tierra orbita con semieje mayor $a = 1$ UA, período $T = 2\pi$ (unidades de tiempo internas) $= 1$ año, y velocidad circular $v_c = 2\pi$ UA/año $\approx 29.78$ km/s.

### 1.3 Transformación al marco geocéntrico

Para el análisis del encuentro cercano también usaremos el **marco geocéntrico**, donde el origen se traslada al centro de la Tierra:

$$\vec{r}_{\text{geo}} = \vec{r}_{\text{Apophis}} - \vec{r}_{\text{Tierra}}$$

$$\vec{v}_{\text{rel}} = \vec{v}_{\text{Apophis}} - \vec{v}_{\text{Tierra}}$$

Este marco es útil para visualizar la trayectoria de Apophis tal como la vería un observador terrestre.

---

## 2. Instalación y configuración

In [ ]:
# Instalar dependencias (ejecutar solo una vez)
!pip install rebound numpy matplotlib plotly -Uq

In [ ]:
import rebound
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configuración global de gráficos
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Constantes físicas en unidades UA-M☉-año
UA_KM   = 1.495978707e8   # 1 UA en km
DL_KM   = 384_400.0       # distancia lunar media en km
RE_KM   = 6_371.0         # radio de la Tierra en km
RE_UA   = RE_KM / UA_KM   # radio de la Tierra en UA
YEAR_S  = 365.25 * 86400  # 1 año en segundos

# Masas en unidades de M☉
M_TIERRA  = 3.00273e-6    # Tierra
M_APOPHIS = 0.0           # Apophis (masa despreciable para perturbar a Sol/Tierra)

print(f"REBOUND versión: {rebound.__version__}")
print(f"1 UA = {UA_KM:.4e} km")
print(f"Radio Tierra = {RE_UA:.4e} UA  ({RE_KM:.0f} km)")
print(f"Distancia lunar = {DL_KM/UA_KM:.4e} UA  ({DL_KM:.0f} km)")

---

## 3. Datos orbitales de Apophis

Los **elementos orbitales keplerianos** de Apophis a la época J2000.0 (JD 2451545.0 = 2000-Jan-01 12:00 TT) son:

| Parámetro | Símbolo | Valor (J2000.0) | Unidad |
|-----------|---------|-----------------|--------|
| Semieje mayor | $a$ | 0.9225787 | UA |
| Excentricidad | $e$ | 0.1912539 | — |
| Inclinación | $i$ | 3.3390 | ° |
| Nodo ascendente | $\Omega$ | 204.0561 | ° |
| Argumento del perihelio | $\omega$ | 126.7690 | ° |
| Anomalía media | $M$ | 201.5477 | ° |

**Fuente:** Farnocchia, D. et al. (2021). *Apophis: The 2029 Earth Encounter*. PSJ 2, 2. DOI: 10.3847/PSJ/abc251

### Clasificación orbital
Apophis pertenece al grupo **Aten**: asteroides con $a < 1$ UA y $Q = a(1+e) > 0.983$ UA (cruzan la órbita de la Tierra). Su período orbital es:

$$T_{\text{Apophis}} = 2\pi\sqrt{\frac{a^3}{GM_\odot}} = 2\pi\sqrt{(0.9226)^3} \approx 2.772 \text{ rad} \approx 323.6 \text{ días}$$

In [ ]:
# ============================================================
# Elementos orbitales de los cuerpos (época J2000.0)
# Fuente: JPL Horizons / Farnocchia et al. (2021)
# ============================================================

# --- Tierra ---
elem_tierra = dict(
    m     = M_TIERRA,
    a     = 1.00000261,
    e     = 0.01671022,
    inc   = np.radians(0.00005),
    Omega = np.radians(348.73936),
    omega = np.radians(102.93735),
    M     = np.radians(100.46435),
    hash  = 'Tierra'
)

# --- Apophis ---
elem_apophis = dict(
    m     = M_APOPHIS,
    a     = 0.9225787,
    e     = 0.1912539,
    inc   = np.radians(3.3390),
    Omega = np.radians(204.0561),
    omega = np.radians(126.7690),
    M     = np.radians(201.5477),
    hash  = 'Apophis'
)

# Periodo de Apophis (unidades REBOUND: tiempo = año/2π)
T_apophis = 2 * np.pi * np.sqrt(elem_apophis['a']**3)
T_tierra  = 2 * np.pi * np.sqrt(elem_tierra['a']**3)

print(f"Período orbital de Apophis: {T_apophis:.4f} años ({T_apophis*365.25:.1f} días)")
print(f"Período orbital de la Tierra: {T_tierra:.4f} años ({T_tierra*365.25:.1f} días)")
print()
print(f"Perihelio de Apophis: q = {elem_apophis['a']*(1-elem_apophis['e']):.4f} UA")
print(f"Afelio  de Apophis: Q = {elem_apophis['a']*(1+elem_apophis['e']):.4f} UA")
print(f"La órbita de Apophis {'CRUZA' if elem_apophis['a']*(1+elem_apophis['e']) > 0.983 else 'NO cruza'} la órbita de la Tierra")

---

## 4. Simulación principal: Sol + Tierra + Apophis

### 4.1 Configuración de REBOUND

Para simular el encuentro de 2029 usamos el integrador **IAS15** (*Implicit integrator with Adaptive Step-Size control, 15th order*), implementado por Rein & Spiegel (2015). Es ideal para:

- Encuentros cercanos (paso de tiempo adaptativo).
- Alta precisión (error de truncamiento ~$10^{-9}$ por período).
- Órbitas muy excéntricas.

**Referencia:** Rein, H. & Spiegel, D.S. (2015). *IAS15: a fast, adaptive, high-order integrator for gravitational dynamics*. MNRAS 446, 1424–1437.

In [ ]:
def crear_simulacion(perturb_venus_marte=False):
    """
    Crea y devuelve una simulación REBOUND del sistema Sol-Tierra-Apophis.
    
    Parámetros
    ----------
    perturb_venus_marte : bool
        Si True, incluye Venus y Marte como perturbadores adicionales.
    
    Sistema de referencia
    ---------------------
    Eclíptico heliocéntrico J2000.0, desplazado al baricentro del sistema.
    Unidades: UA, M☉, año (con G = 4π²).
    """
    sim = rebound.Simulation()
    sim.G          = 4 * np.pi**2   # UA³/(M☉·año²)
    sim.integrator = 'ias15'        # adaptativo, orden 15
    sim.dt         = 1e-3           # paso inicial: ~0.365 días

    # Sol — en el origen del marco heliocéntrico
    sim.add(m=1.0, hash='Sol')

    # Tierra — elementos orbitales J2000.0
    sim.add(**elem_tierra)

    if perturb_venus_marte:
        # Venus (perturbador gravitacional importante)
        sim.add(m=2.4478e-6, a=0.72333, e=0.00677,
                inc=np.radians(3.395), Omega=np.radians(76.680),
                omega=np.radians(131.602), M=np.radians(50.115),
                hash='Venus')
        # Marte
        sim.add(m=3.2271e-7, a=1.52366, e=0.09340,
                inc=np.radians(1.850), Omega=np.radians(49.562),
                omega=np.radians(336.060), M=np.radians(355.453),
                hash='Marte')

    # Apophis — elementos orbitales J2000.0
    sim.add(**elem_apophis)

    # Mover al baricentro del sistema (corrección pequeña pero correcta)
    sim.move_to_com()
    return sim


# Verificar posiciones iniciales (t=0 = J2000.0)
sim_test = crear_simulacion()
print("Posiciones iniciales en J2000.0 (marco baricéntrico, unidades UA):")
print(f"  Sol     : x={sim_test.particles['Sol'].x:.6f}  y={sim_test.particles['Sol'].y:.6f}  z={sim_test.particles['Sol'].z:.6f}")
print(f"  Tierra  : x={sim_test.particles['Tierra'].x:.6f}  y={sim_test.particles['Tierra'].y:.6f}  z={sim_test.particles['Tierra'].z:.6f}")
print(f"  Apophis : x={sim_test.particles['Apophis'].x:.6f}  y={sim_test.particles['Apophis'].y:.6f}  z={sim_test.particles['Apophis'].z:.6f}")

# Distancia inicial Tierra-Apophis
dt0 = sim_test.particles['Tierra']
da0 = sim_test.particles['Apophis']
d0 = np.sqrt((dt0.x-da0.x)**2 + (dt0.y-da0.y)**2 + (dt0.z-da0.z)**2)
print(f"\nDistancia inicial Tierra-Apophis: {d0:.4f} UA = {d0*UA_KM:.0f} km")

### 4.2 Integración: búsqueda del encuentro cercano de 2029

Integramos desde J2000.0 hasta ~2031, registrando la posición de cada cuerpo cada día. La aproximación de 2029 ocurre alrededor de $t \approx 29.28$ años desde J2000.0 (13 de abril de 2029).

In [ ]:
# ============================================================
# Integración 30 años — muestreo diario
# ============================================================

sim = crear_simulacion(perturb_venus_marte=False)

T_INT  = 30.0          # años totales de integración
DT_OUT = 1.0 / 365.25  # muestreo: 1 día en años
N_OUT  = int(T_INT / DT_OUT) + 1
t_arr  = np.linspace(0, T_INT, N_OUT)

# Arrays para almacenar posiciones y velocidades (UA y UA/año)
r = {'Sol': np.zeros((N_OUT,3)),
     'Tierra': np.zeros((N_OUT,3)),
     'Apophis': np.zeros((N_OUT,3))}
v = {'Tierra': np.zeros((N_OUT,3)),
     'Apophis': np.zeros((N_OUT,3))}

for i, t in enumerate(t_arr):
    sim.integrate(t)
    for name in ['Sol', 'Tierra', 'Apophis']:
        p = sim.particles[name]
        r[name][i] = [p.x, p.y, p.z]
        if name != 'Sol':
            v[name][i] = [p.vx, p.vy, p.vz]

# ============================================================
# Calcular distancias y velocidades relativas
# ============================================================

# Distancia Tierra-Apophis (UA)
dr     = r['Apophis'] - r['Tierra']
d_ua   = np.linalg.norm(dr, axis=1)
d_km   = d_ua * UA_KM
d_dl   = d_km / DL_KM    # en distancias lunares

# Velocidad relativa Apophis-Tierra (km/s)
dv        = v['Apophis'] - v['Tierra']
vrel_ua_y = np.linalg.norm(dv, axis=1)         # UA/año
vrel_kms  = vrel_ua_y * UA_KM / YEAR_S         # km/s

# Distancia heliocentrica de Apophis (UA)
d_apo_sol = np.linalg.norm(r['Apophis'] - r['Sol'], axis=1)

# ============================================================
# Encontrar el encuentro más cercano
# ============================================================
idx_min  = np.argmin(d_ua)
t_min    = t_arr[idx_min]
d_min_ua = d_ua[idx_min]
d_min_km = d_km[idx_min]
d_min_dl = d_dl[idx_min]
v_enc    = vrel_kms[idx_min]

print("=" * 60)
print("     ENCUENTRO MÁS CERCANO — APOPHIS 2029")
print("=" * 60)
print(f"  t desde J2000.0  : {t_min:.4f} años  ({t_min*365.25:.1f} días)")
print(f"  Año aproximado   : {2000 + t_min:.4f}  ≈ año 2029")
print(f"  Distancia mínima : {d_min_ua:.6f} UA")
print(f"                     {d_min_km:.0f} km")
print(f"                     {d_min_dl:.3f} distancias lunares")
print(f"  Velocidad relat. : {v_enc:.2f} km/s")
print("=" * 60)
print()
print(f"  Valor oficial (JPL): ~38,017 km  ({38017/DL_KM:.3f} LD)")
print(f"  Discrepancia       : {abs(d_min_km - 38017):.0f} km")
print()
print("Nota: la discrepancia es esperada porque usamos elementos")
print("keplerianos aproximados sin correcciones relativistas ni")
print("el efecto Yarkovsky. Ver Sec. 7 para la simulación con")
print("condiciones iniciales reales de NASA Horizons.")

---

## 5. Visualizaciones

### 5.1 Gráfico 1 — Órbitas heliocentricas en el plano de la eclíptica (2D)

In [ ]:
# ============================================================
# Simulación de una vuelta completa de cada cuerpo para
# graficar las órbitas (sin necesidad de 30 años)
# ============================================================

sim_orb = crear_simulacion()
T_plot  = max(T_tierra, T_apophis) * 2   # 2 períodos del más lento
N_orb   = 2000
t_orb   = np.linspace(0, T_plot, N_orb)

xT, yT, zT = [], [], []
xA, yA, zA = [], [], []

for t in t_orb:
    sim_orb.integrate(t)
    pt = sim_orb.particles['Tierra']
    pa = sim_orb.particles['Apophis']
    xT.append(pt.x); yT.append(pt.y); zT.append(pt.z)
    xA.append(pa.x); yA.append(pa.y); zA.append(pa.z)

xT, yT, zT = np.array(xT), np.array(yT), np.array(zT)
xA, yA, zA = np.array(xA), np.array(yA), np.array(zA)

# ============================================================
# Gráfico 1: Órbitas en el plano de la eclíptica
# ============================================================

fig, ax = plt.subplots(figsize=(9, 9))

# Línea de intersección de los planos orbitales
theta_nodo = np.radians(elem_apophis['Omega'])
ax.axline((0, 0), (np.cos(theta_nodo), np.sin(theta_nodo)),
          color='gray', lw=0.8, ls=':', alpha=0.6, label='Nodo ascendente Apophis')

# Órbita de la Tierra
ax.plot(xT, yT, color='royalblue', lw=2.5, label='Órbita Tierra')

# Órbita de Apophis (con gradiente de color para mostrar el movimiento)
points_A = np.array([xA, yA]).T.reshape(-1, 1, 2)
segs_A   = np.concatenate([points_A[:-1], points_A[1:]], axis=1)
norm_A   = Normalize(vmin=0, vmax=N_orb)
cmap_A   = plt.cm.plasma
lc_A     = plt.matplotlib.collections.LineCollection(segs_A, cmap=cmap_A, norm=norm_A, lw=2, alpha=0.9)
lc_A.set_array(np.arange(N_orb))
ax.add_collection(lc_A)
plt.colorbar(lc_A, ax=ax, label='Paso de tiempo (index)', fraction=0.03, pad=0.02)

# Sol
ax.scatter([0], [0], s=350, color='gold', edgecolors='darkorange', lw=2,
           zorder=10, label='Sol')

# Marcadores de posición inicial
ax.scatter([xT[0]], [yT[0]], s=100, color='royalblue', edgecolors='navy',
           lw=2, zorder=10, label='Tierra t=J2000.0')
ax.scatter([xA[0]], [yA[0]], s=100, color='tomato', edgecolors='darkred',
           lw=2, zorder=10, label='Apophis t=J2000.0')

# Perihelio y afelio de Apophis
q_A = elem_apophis['a'] * (1 - elem_apophis['e'])  # perihelio
Q_A = elem_apophis['a'] * (1 + elem_apophis['e'])  # afelio
ax.annotate(f'Perihelio\n({q_A:.3f} UA)', xy=(q_A*0.98, 0.04),
            fontsize=8.5, color='darkred', ha='right')
ax.annotate(f'Afelio\n({Q_A:.3f} UA)', xy=(-Q_A*0.97, 0.04),
            fontsize=8.5, color='darkred', ha='left')

# Posición de Apophis en el encuentro (t_min)
sim_orb2 = crear_simulacion()
sim_orb2.integrate(t_min)
xA_enc = sim_orb2.particles['Apophis'].x
yA_enc = sim_orb2.particles['Apophis'].y
xT_enc = sim_orb2.particles['Tierra'].x
yT_enc = sim_orb2.particles['Tierra'].y

ax.scatter([xA_enc], [yA_enc], s=160, color='red', marker='*', zorder=12,
           edgecolors='white', lw=1, label=f'Apophis en encuentro ({2000+t_min:.1f})')
ax.scatter([xT_enc], [yT_enc], s=160, color='deepskyblue', marker='*', zorder=12,
           edgecolors='white', lw=1, label=f'Tierra en encuentro')
ax.annotate('', xy=(xA_enc, yA_enc), xytext=(xT_enc, yT_enc),
            arrowprops=dict(arrowstyle='<->', color='lime', lw=1.5))

# Ejes y formato
ax.set_aspect('equal')
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_xlabel('x [UA] — dirección al equinoccio vernal ♈ (J2000.0)', fontsize=11)
ax.set_ylabel('y [UA] — en el plano de la eclíptica', fontsize=11)
ax.set_title('Órbitas heliocentricas de Tierra y Apophis\n'
             '(Marco eclíptico J2000.0 — proyección en el plano $xy$)', fontsize=13)
ax.legend(loc='upper right', fontsize=9)

# Círculo de la esfera de influencia de la Tierra (Hill sphere)
r_hill = 1.0 * (M_TIERRA / 3)**(1/3)
circ_hill = plt.Circle((xT_enc, yT_enc), r_hill, fill=False,
                        edgecolor='deepskyblue', lw=1.2, ls='--', alpha=0.7)
ax.add_patch(circ_hill)
ax.annotate(f'Esfera de Hill\n({r_hill*UA_KM:.0f} km)', 
            xy=(xT_enc + r_hill*0.7, yT_enc + r_hill*0.7),
            fontsize=8, color='deepskyblue')

plt.tight_layout()
plt.savefig('/tmp/orbitas_heliocentricas.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Radio esfera de Hill de la Tierra: {r_hill:.5f} UA = {r_hill*UA_KM:.0f} km")

### 5.2 Gráfico 2 — Órbitas en 3D (mostrando la inclinación orbital de Apophis)

In [ ]:
# ============================================================
# Gráfico 2: Órbitas 3D — perspectiva inclinada
# ============================================================

fig = plt.figure(figsize=(12, 9))
ax3 = fig.add_subplot(111, projection='3d')

# Órbita de la Tierra (prácticamente en el plano z=0)
ax3.plot(xT, yT, zT, color='royalblue', lw=2, label='Órbita Tierra', alpha=0.9)

# Órbita de Apophis (con inclinación 3.34°)
ax3.plot(xA, yA, zA, color='tomato', lw=2, ls='--', label='Órbita Apophis', alpha=0.9)

# Sol
ax3.scatter([0], [0], [0], s=300, color='gold', edgecolors='darkorange',
            lw=2, zorder=10, label='Sol')

# Posición en el encuentro
ax3.scatter([xA_enc], [yA_enc], [sim_orb2.particles['Apophis'].z],
            s=200, color='red', marker='*', zorder=12, label='Apophis — encuentro')
ax3.scatter([xT_enc], [yT_enc], [sim_orb2.particles['Tierra'].z],
            s=200, color='deepskyblue', marker='*', zorder=12, label='Tierra — encuentro')

# Proyección de la órbita de Apophis en z=0
ax3.plot(xA, yA, np.zeros_like(zA), color='salmon', lw=0.8, ls=':', alpha=0.4)

# Líneas verticales indicando el desfase en z
nodos = np.where(np.diff(np.sign(zA)))[0]
for n in nodos[:4]:
    ax3.plot([xA[n], xA[n]], [yA[n], yA[n]], [0, zA[n]],
             color='gray', lw=0.8, ls='--', alpha=0.5)

ax3.set_xlabel('x [UA]  →  equinoccio vernal ♈', labelpad=10)
ax3.set_ylabel('y [UA]', labelpad=10)
ax3.set_zlabel('z [UA]  →  polo norte eclíptico', labelpad=10)
ax3.set_title('Visualización 3D — Órbitas de Tierra y Apophis\n'
              f'Inclinación de Apophis: i = {np.degrees(elem_apophis["inc"]):.2f}°', fontsize=13)
ax3.legend(loc='upper left', fontsize=9)
ax3.view_init(elev=25, azim=60)

plt.tight_layout()
plt.savefig('/tmp/orbitas_3d.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Gráfico 3 — Distancia Tierra-Apophis vs tiempo (30 años)

In [ ]:
# ============================================================
# Gráfico 3: Distancia Tierra-Apophis durante 30 años
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
t_years = 2000 + t_arr   # convertir a años calendario

# --- Panel izquierdo: vista completa (30 años) ---
ax = axes[0]
ax.plot(t_years, d_dl, color='steelblue', lw=0.8, alpha=0.85)
ax.axhline(1,  color='gray',   ls='--', lw=1.2, label='1 LD — órbita lunar')
ax.axhline(10, color='orange', ls='--', lw=1.2, label='10 LD ≈ 3.84 Mkm')
ax.axhline(0.25, color='red', ls=':', lw=1.5, label='0.25 LD — distancia del encuentro')
ax.axvline(2000 + t_min, color='crimson', ls='-', lw=1.8,
           label=f'Encuentro: {2000+t_min:.3f} ({d_min_dl:.2f} LD)')

# Marcar los mínimos locales (aproximaciones sucesivas)
from scipy.signal import argrelmin
try:
    local_mins = argrelmin(d_ua, order=50)[0]
    local_mins_close = local_mins[d_dl[local_mins] < 100]
    ax.scatter(t_years[local_mins_close], d_dl[local_mins_close],
               color='orange', s=50, zorder=5, label='Aproximaciones locales')
except ImportError:
    pass

ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('Distancia Tierra-Apophis [distancias lunares]', fontsize=11)
ax.set_title('Distancia Tierra-Apophis (2000–2030)', fontsize=13)
ax.set_ylim(0, 400)
ax.legend(fontsize=9)

# --- Panel derecho: zoom alrededor del encuentro ---
ax = axes[1]
ventana = 0.3  # ±0.3 años alrededor del mínimo
mask = np.abs(t_arr - t_min) < ventana

# Colorear la línea según la velocidad relativa
t_z = t_years[mask]
d_z = d_dl[mask]
v_z = vrel_kms[mask]

points_z = np.array([t_z, d_z]).T.reshape(-1, 1, 2)
segs_z   = np.concatenate([points_z[:-1], points_z[1:]], axis=1)
norm_z   = Normalize(vmin=v_z.min(), vmax=v_z.max())
lc_z     = plt.matplotlib.collections.LineCollection(segs_z, cmap='hot_r',
                                                     norm=norm_z, lw=3)
lc_z.set_array(v_z)
ax.add_collection(lc_z)
plt.colorbar(lc_z, ax=ax, label='Velocidad relativa [km/s]', fraction=0.04, pad=0.02)

ax.axhline(1,  color='gray', ls='--', lw=1.2, label='1 LD')
ax.axhline(d_min_dl, color='crimson', ls=':', lw=1.5,
           label=f'Mínimo: {d_min_dl:.3f} LD\n({d_min_km:.0f} km)')
ax.axvline(2000+t_min, color='crimson', ls='--', lw=1.5)

# Zona de satélites geoestacionarios (~6.6 radios terrestres)
r_geo_dl = 42_164 / DL_KM
ax.axhspan(0, r_geo_dl, color='cyan', alpha=0.12, label=f'Debajo de GEO ({r_geo_dl:.3f} LD)')

ax.set_xlim(2000+t_min-ventana, 2000+t_min+ventana)
ax.set_ylim(-0.02, max(d_z)*1.05)
ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('Distancia Tierra-Apophis [distancias lunares]', fontsize=11)
ax.set_title(f'Zoom: encuentro cercano 2029\nt_min ≈ {2000+t_min:.4f}', fontsize=13)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/distancia_vs_tiempo.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Gráfico 4 — Trayectoria geocéntrica de Apophis (marco centrado en la Tierra)

In [ ]:
# ============================================================
# Transformación al marco geocéntrico
# r_geo = r_Apophis - r_Tierra
# ============================================================

# Solo la ventana temporal alrededor del encuentro (±2 días)
ventana_geo = 5.0 / 365.25   # 5 días
mask_geo    = np.abs(t_arr - t_min) < ventana_geo

r_geo = (r['Apophis'] - r['Tierra'])[mask_geo]  # UA
t_geo = t_arr[mask_geo]
d_geo = np.linalg.norm(r_geo, axis=1) * UA_KM   # km
v_geo_kms = vrel_kms[mask_geo]

# Convertir a km para el gráfico geocéntrico
r_geo_km = r_geo * UA_KM

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Panel izquierdo: proyección en el plano xy (eclíptica geocéntrica) ---
ax = axes[0]
# Colorear según el tiempo
norm_t = Normalize(vmin=t_geo.min(), vmax=t_geo.max())
cmap_t = plt.cm.coolwarm
points_g = np.array([r_geo_km[:,0], r_geo_km[:,1]]).T.reshape(-1,1,2)
segs_g   = np.concatenate([points_g[:-1], points_g[1:]], axis=1)
lc_g     = plt.matplotlib.collections.LineCollection(segs_g, cmap=cmap_t, norm=norm_t, lw=3)
lc_g.set_array(t_geo)
ax.add_collection(lc_g)
cb_g = plt.colorbar(lc_g, ax=ax, fraction=0.04, pad=0.02)
cb_g.set_label('Tiempo (años desde J2000)')

# Tierra en el origen
ax.scatter([0], [0], s=300, color='royalblue', edgecolors='navy',
           lw=2, zorder=10, label='Tierra (origen geocéntrico)')

# Satélites geoestacionarios
circ_geo = plt.Circle((0,0), 42164, fill=False, edgecolor='cyan',
                      lw=1.2, ls='--', label='Órbita GEO (42,164 km)')
ax.add_patch(circ_geo)

# Punto de máxima aproximación
idx_min_geo = np.argmin(d_geo)
ax.scatter([r_geo_km[idx_min_geo,0]], [r_geo_km[idx_min_geo,1]],
           s=200, color='red', marker='*', zorder=12, edgecolors='white', lw=1,
           label=f'Mín. ({d_geo[idx_min_geo]:.0f} km)')

# Dirección de llegada y salida
ax.annotate('Entrada', xy=(r_geo_km[0,0], r_geo_km[0,1]),
            fontsize=9, color='steelblue', ha='center')
ax.annotate('Salida', xy=(r_geo_km[-1,0], r_geo_km[-1,1]),
            fontsize=9, color='crimson', ha='center')

ax.set_aspect('equal')
margin = max(np.abs(r_geo_km[:,:2]).max()) * 1.2
ax.set_xlim(-margin, margin)
ax.set_ylim(-margin, margin)
ax.set_xlabel('Δx geocéntrico [km]', fontsize=11)
ax.set_ylabel('Δy geocéntrico [km]', fontsize=11)
ax.set_title('Trayectoria de Apophis en el marco geocéntrico\n'
             '(proyección en el plano eclíptico, ±5 días)', fontsize=12)
ax.legend(fontsize=9, loc='upper right')

# --- Panel derecho: distancia geocéntrica en km vs tiempo ---
ax = axes[1]
t_geo_days = (t_geo - t_min) * 365.25   # días desde el encuentro

ax.plot(t_geo_days, d_geo, color='crimson', lw=2.5, label='Distancia Apophis-Tierra')
ax.fill_between(t_geo_days, 0, d_geo, color='salmon', alpha=0.2)

# Líneas de referencia
for label_r, val_km, col in [
    ('Superficie terrestre', RE_KM, 'blue'),
    ('Órbita ISS (~400 km)', 400 + RE_KM, 'green'),
    ('Órbita GEO (42,164 km)', 42164, 'cyan'),
    ('Distancia lunar (384,400 km)', DL_KM, 'gray'),
]:
    ax.axhline(val_km, color=col, ls='--', lw=1.2, alpha=0.7, label=label_r)

ax.axvline(0, color='black', ls=':', lw=1)
ax.scatter([0], [d_geo[idx_min_geo]], s=120, color='red', zorder=10)
ax.annotate(f'{d_geo[idx_min_geo]:.0f} km\n({d_geo[idx_min_geo]/DL_KM:.3f} LD)',
            xy=(0, d_geo[idx_min_geo]),
            xytext=(1.5, d_geo[idx_min_geo]*3),
            fontsize=9, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_yscale('log')
ax.set_xlabel('Días desde el encuentro más cercano', fontsize=11)
ax.set_ylabel('Distancia Tierra-Apophis [km] (escala log)', fontsize=11)
ax.set_title('Distancia geocéntrica de Apophis\n(escala logarítmica)', fontsize=12)
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('/tmp/trayectoria_geocentrica.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Gráfico 5 — Velocidad relativa de Apophis respecto a la Tierra

In [ ]:
# ============================================================
# Gráfico 5: Velocidad relativa y componentes
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ventana_v  = 15.0 / 365.25   # ±15 días
mask_v     = np.abs(t_arr - t_min) < ventana_v
t_v_days   = (t_arr[mask_v] - t_min) * 365.25
vrel_v     = vrel_kms[mask_v]
dv_v       = dv[mask_v] * UA_KM / YEAR_S   # componentes en km/s
d_v_km     = d_km[mask_v]

# Panel (0,0): Módulo de velocidad relativa vs tiempo
ax = axes[0,0]
ax.plot(t_v_days, vrel_v, 'tomato', lw=2.5)
ax.axvline(0, color='black', ls=':', lw=1)
ax.scatter([0], [v_enc], s=100, color='red', zorder=10)
ax.annotate(f'v_enc = {v_enc:.2f} km/s', xy=(0, v_enc),
            xytext=(3, v_enc+0.3), fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))
ax.set_xlabel('Días desde el encuentro', fontsize=10)
ax.set_ylabel('|v_rel| [km/s]', fontsize=10)
ax.set_title('Módulo de la velocidad relativa Apophis-Tierra', fontsize=11)

# Panel (0,1): Componentes de la velocidad relativa
ax = axes[0,1]
colors_v = ['royalblue', 'seagreen', 'tomato']
labels_v = ['$v_x$ (eclíptica, hacia ♈)', '$v_y$ (eclíptica)', '$v_z$ (hacia polo)']
for j, (col, lab) in enumerate(zip(colors_v, labels_v)):
    ax.plot(t_v_days, dv_v[:,j], color=col, lw=1.8, label=lab)
ax.axvline(0, color='black', ls=':', lw=1)
ax.axhline(0, color='black', ls='-', lw=0.5, alpha=0.4)
ax.set_xlabel('Días desde el encuentro', fontsize=10)
ax.set_ylabel('Componentes $v_{rel}$ [km/s]', fontsize=10)
ax.set_title('Componentes de la velocidad relativa', fontsize=11)
ax.legend(fontsize=9)

# Panel (1,0): Diagrama velocidad vs distancia (hodógrafo de velocidad-distancia)
ax = axes[1,0]
sc = ax.scatter(d_v_km, vrel_v, c=t_v_days, cmap='coolwarm', s=15, alpha=0.8)
plt.colorbar(sc, ax=ax, label='Días desde encuentro')
ax.axvline(d_min_km, color='red', ls='--', lw=1.5, label=f'd_min={d_min_km:.0f} km')
ax.set_xlabel('Distancia Tierra-Apophis [km]', fontsize=10)
ax.set_ylabel('|v_rel| [km/s]', fontsize=10)
ax.set_title('Velocidad vs. distancia\n(entrada y salida del encuentro)', fontsize=11)
ax.legend(fontsize=9)

# Panel (1,1): Energía de la hipérbola geocéntrica
# E = v²/2 - GM_tierra/r; si E > 0 la trayectoria es hiperbólica
GM_tierra = 4 * np.pi**2 * M_TIERRA  # UA³/año²
GM_tierra_km3_s2 = 3.986e5  # km³/s²
d_v_km_arr = d_v_km
E_hyp = 0.5 * vrel_v**2 - GM_tierra_km3_s2 / d_v_km_arr  # km²/s²

ax = axes[1,1]
ax.plot(t_v_days, E_hyp, 'purple', lw=2)
ax.axhline(0, color='black', ls='--', lw=1)
ax.fill_between(t_v_days, E_hyp, 0, where=(E_hyp>0), color='salmon', alpha=0.3,
                label='E > 0 (hiperbólica — no ligada)')
ax.fill_between(t_v_days, E_hyp, 0, where=(E_hyp<0), color='lightblue', alpha=0.3,
                label='E < 0 (elíptica — ligada)')
ax.axvline(0, color='black', ls=':', lw=1)
ax.set_xlabel('Días desde el encuentro', fontsize=10)
ax.set_ylabel('Energía específica geocéntrica [km²/s²]', fontsize=10)
ax.set_title('Energía geocéntrica de Apophis\n'
             '(positivo = trayectoria hiperbólica)', fontsize=11)
ax.legend(fontsize=9)

plt.suptitle('Análisis de la velocidad relativa — Apophis vs. Tierra (±15 días)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/velocidad_relativa.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 Gráfico 6 — Conservación de energía y momento angular

In [ ]:
# ============================================================
# Gráfico 6: Conservación de la energía y momento angular
# del sistema Sol-Tierra-Apophis
# ============================================================

sim_cons = crear_simulacion()

# Energía y momento angular iniciales
E0  = sim_cons.calculate_energy()
L0  = np.array(sim_cons.calculate_angular_momentum())   # vector L
Lz0 = L0[2]

N_cons  = 3000
t_cons  = np.linspace(0, T_INT, N_cons)
dE_rel  = np.zeros(N_cons)
dLz_rel = np.zeros(N_cons)

for i, t in enumerate(t_cons):
    sim_cons.integrate(t)
    Ei   = sim_cons.calculate_energy()
    Li   = np.array(sim_cons.calculate_angular_momentum())
    dE_rel[i]  = abs((Ei  - E0)  / E0)
    dLz_rel[i] = abs((Li[2] - Lz0) / Lz0) if Lz0 != 0 else 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(2000 + t_cons, dE_rel, color='crimson', lw=1.5)
ax.axhline(1e-10, color='gray', ls='--', lw=1, label='10⁻¹⁰ (límite máquina)')
ax.axvline(2000+t_min, color='blue', ls='--', lw=1.2, label=f'Encuentro 2029')
ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('|ΔE/E₀|', fontsize=11)
ax.set_title('Error relativo en la energía total del sistema\n(IAS15, 30 años)', fontsize=12)
ax.legend(fontsize=9)

ax = axes[1]
ax.semilogy(2000 + t_cons, dLz_rel + 1e-18, color='steelblue', lw=1.5)
ax.axhline(1e-10, color='gray', ls='--', lw=1, label='10⁻¹⁰')
ax.axvline(2000+t_min, color='blue', ls='--', lw=1.2, label=f'Encuentro 2029')
ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('|ΔLz/Lz₀|', fontsize=11)
ax.set_title('Error relativo en la componente z del momento angular\n(IAS15, 30 años)', fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/conservacion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Error máximo en energía: {dE_rel.max():.2e}")
print(f"Error máximo en Lz:      {dLz_rel.max():.2e}")

### 5.7 Gráfico 7 — Efecto de Venus y Marte en la distancia mínima

In [ ]:
# ============================================================
# Gráfico 7: Comparación con y sin perturbadores (Venus+Marte)
# Muestra cómo las perturbaciones afectan la distancia mínima
# ============================================================

# Simulación sin perturbadores (ya calculada: d_ua)
# Simulación con perturbadores
print("Simulando con Venus y Marte como perturbadores... (tarda ~1-2 min)")

sim_pert = crear_simulacion(perturb_venus_marte=True)

# Solo integrar alrededor del encuentro (±2 años)
t_ini_pert = max(0, t_min - 2.0)
t_fin_pert = t_min + 2.0
N_pert     = 3000
t_pert     = np.linspace(t_ini_pert, t_fin_pert, N_pert)

d_ua_pert = np.zeros(N_pert)
for i, t in enumerate(t_pert):
    sim_pert.integrate(t)
    pt = sim_pert.particles['Tierra']
    pa = sim_pert.particles['Apophis']
    d_ua_pert[i] = np.sqrt((pt.x-pa.x)**2 + (pt.y-pa.y)**2 + (pt.z-pa.z)**2)

d_km_pert = d_ua_pert * UA_KM
d_dl_pert = d_km_pert / DL_KM
t_pert_years = 2000 + t_pert

idx_min_pert = np.argmin(d_ua_pert)
d_min_pert   = d_km_pert[idx_min_pert]
t_min_pert   = t_pert[idx_min_pert]

# Comparación
mask_comp = np.abs(t_arr - t_min) < 2.0

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(2000 + t_arr[mask_comp], d_dl[mask_comp],
        'steelblue', lw=2.5, label='Sol + Tierra + Apophis (3 cuerpos)')
ax.plot(t_pert_years, d_dl_pert,
        'tomato', lw=2.5, ls='--', label='Sol + Venus + Tierra + Marte + Apophis (5 cuerpos)')

ax.axhline(1, color='gray', ls=':', lw=1, label='1 LD')
ax.axvline(2000 + t_min,      color='steelblue', ls='--', lw=1.5,
           label=f'3 cuerpos: {d_min_km:.0f} km ({d_dl[idx_min]:.3f} LD)')
ax.axvline(2000 + t_min_pert, color='tomato', ls='--', lw=1.5,
           label=f'5 cuerpos: {d_min_pert:.0f} km ({d_dl_pert[idx_min_pert]:.3f} LD)')

ax.set_xlabel('Año', fontsize=11)
ax.set_ylabel('Distancia Tierra-Apophis [distancias lunares]', fontsize=11)
ax.set_title('Efecto de Venus y Marte sobre la trayectoria de Apophis\n'
             '(comparación 3 cuerpos vs. 5 cuerpos)', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 15)

plt.tight_layout()
plt.savefig('/tmp/perturbadores.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDist. mínima (3 cuerpos) : {d_min_km:.0f} km  ({d_dl[idx_min]:.4f} LD)")
print(f"Dist. mínima (5 cuerpos) : {d_min_pert:.0f} km  ({d_dl_pert[idx_min_pert]:.4f} LD)")
print(f"Diferencia               : {abs(d_min_km - d_min_pert):.0f} km")

---

## 6. Simulación con condiciones iniciales reales de NASA Horizons

REBOUND puede descargar condiciones iniciales directamente de la base de datos **JPL Horizons** del Jet Propulsion Laboratory de la NASA. Esto nos da los vectores de estado $(\vec{r}, \vec{v})$ calculados con el modelo de alta precisión de la NASA, que incluye:

- Todos los planetas
- Efecto Yarkovsky (fuerza de radiación asimétrica)
- Correcciones relativistas
- Perturbaciones de la Luna

**Referencia:** Giorgini, J.D. et al. (1996). *JPL's On-Line Solar System Data Service*. Bulletin of the AAS 28, 1158.

Usamos la fecha **2028-Jan-01** como época de inicio (cercana al encuentro de 2029).

In [ ]:
# ============================================================
# Simulación con NASA Horizons
# Requiere conexión a internet
# ============================================================

import ssl
import rebound.horizons

# Desactivar verificación SSL (necesario en algunos entornos)
rebound.horizons.SSL_CONTEXT = 'unverified'

try:
    FECHA_HORIZONS = '2028-01-01'
    sim_h = rebound.Simulation()
    sim_h.G          = 4 * np.pi**2
    sim_h.integrator = 'ias15'

    print(f"Descargando datos de NASA Horizons para {FECHA_HORIZONS}...")
    sim_h.add('Sun',    date=FECHA_HORIZONS)
    sim_h.add('Earth',  date=FECHA_HORIZONS)
    sim_h.add('Apophis',date=FECHA_HORIZONS)   # ID oficial: 99942
    sim_h.move_to_com()

    print("Datos descargados exitosamente.")
    print(f"  Partículas en la simulación: {sim_h.N}")
    for p in sim_h.particles:
        print(f"  {p.hash.value:>12s}: m = {p.m:.3e} M☉  "
              f"r = ({p.x:.4f}, {p.y:.4f}, {p.z:.4f}) UA")

    HORIZONS_OK = True

except Exception as e:
    print(f"Error al conectar con NASA Horizons: {e}")
    print("Continuando con los elementos orbitales aproximados.")
    HORIZONS_OK = False

In [ ]:
if HORIZONS_OK:
    # ============================================================
    # Integrar desde 2028-Jan-01 hasta 2029-Jun-01 (497 días)
    # ============================================================
    T_H  = 497.0 / 365.25   # en años
    N_H  = 2000
    t_H  = np.linspace(0, T_H, N_H)

    dH_km = np.zeros(N_H)
    vH_ks = np.zeros(N_H)

    for i, t in enumerate(t_H):
        sim_h.integrate(t)
        pt = sim_h.particles[1]   # Tierra
        pa = sim_h.particles[2]   # Apophis
        dH_km[i] = np.sqrt((pt.x-pa.x)**2 + (pt.y-pa.y)**2 + (pt.z-pa.z)**2) * UA_KM
        dvx = (pa.vx - pt.vx) * UA_KM / YEAR_S
        dvy = (pa.vy - pt.vy) * UA_KM / YEAR_S
        dvz = (pa.vz - pt.vz) * UA_KM / YEAR_S
        vH_ks[i] = np.sqrt(dvx**2 + dvy**2 + dvz**2)

    idx_H   = np.argmin(dH_km)
    t_H_enc = 2028 + t_H[idx_H] * 365.25 / 365.25   # fracción de año

    print("=" * 60)
    print("   RESULTADO CON NASA HORIZONS")
    print("=" * 60)
    print(f"  Distancia mínima : {dH_km[idx_H]:.0f} km")
    print(f"                     {dH_km[idx_H]/DL_KM:.4f} distancias lunares")
    print(f"  Velocidad enc.   : {vH_ks[idx_H]:.2f} km/s")
    print(f"  Año del encuentro: {t_H_enc:.4f}")
    print("="*60)
    print(f"  Valor oficial JPL: ~38,017 km")

    # Gráfico comparativo Horizons vs Keplerian
    fig, ax = plt.subplots(figsize=(11, 5))
    t_H_days = t_H * 365.25   # días desde 2028-Jan-01

    ax.plot(t_H_days, dH_km / DL_KM, 'tomato', lw=2.5, label='NASA Horizons (real)')

    # Superponer los datos de la simulación con elementos keplerianos
    mask_comp2 = (t_arr >= t_min - 1.5) & (t_arr <= t_min + 0.5)
    t_kepl_days = (t_arr[mask_comp2] - (t_min - 1.5)) * 365.25  # alinear con horizons
    ax.plot(t_kepl_days + (t_H[idx_H] - 1.5)*365.25 ,
            d_dl[mask_comp2], 'steelblue', lw=2, ls='--',
            label='Elementos keplerianos J2000.0')

    ax.axhline(1, color='gray', ls=':', lw=1, label='1 LD')
    ax.axhline(38017/DL_KM, color='limegreen', ls='--', lw=1.5,
               label=f'Oficial JPL: 38,017 km ({38017/DL_KM:.4f} LD)')
    ax.set_xlabel('Días desde 2028-Jan-01', fontsize=11)
    ax.set_ylabel('Distancia Tierra-Apophis [LD]', fontsize=11)
    ax.set_title('Comparación: NASA Horizons vs. elementos orbitales keplerianos', fontsize=13)
    ax.set_ylim(0, 5)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("Sección de Horizons omitida (sin conexión).")
    print("Para ejecutar esta sección, asegúrese de tener acceso a internet.")

---

## 7. Gráfico interactivo con Plotly

In [ ]:
# ============================================================
# Gráfico 8: Visualización interactiva 3D con Plotly
# ============================================================

fig_p = go.Figure()

# Órbita de la Tierra
fig_p.add_trace(go.Scatter3d(
    x=xT, y=yT, z=zT,
    mode='lines',
    name='Órbita Tierra',
    line=dict(color='royalblue', width=4)
))

# Órbita de Apophis (coloreada por tiempo)
fig_p.add_trace(go.Scatter3d(
    x=xA, y=yA, z=zA,
    mode='lines',
    name='Órbita Apophis',
    line=dict(color='tomato', width=4, dash='dash')
))

# Sol
fig_p.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    name='Sol',
    marker=dict(size=12, color='gold', symbol='circle',
                line=dict(color='orange', width=2))
))

# Tierra en el encuentro
fig_p.add_trace(go.Scatter3d(
    x=[xT_enc], y=[yT_enc], z=[sim_orb2.particles['Tierra'].z],
    mode='markers',
    name=f'Tierra — encuentro ({2000+t_min:.2f})',
    marker=dict(size=8, color='deepskyblue', symbol='circle')
))

# Apophis en el encuentro
fig_p.add_trace(go.Scatter3d(
    x=[xA_enc], y=[yA_enc], z=[sim_orb2.particles['Apophis'].z],
    mode='markers',
    name=f'Apophis — encuentro',
    marker=dict(size=8, color='red', symbol='diamond')
))

fig_p.update_layout(
    title=dict(
        text='Visualización interactiva 3D — Sistema Sol-Tierra-Apophis<br>'
             '<sup>Marco eclíptico heliocéntrico J2000.0</sup>',
        x=0.5, font=dict(size=16)
    ),
    scene=dict(
        xaxis_title='x [UA]  →  ♈ vernal',
        yaxis_title='y [UA]',
        zaxis_title='z [UA]  →  polo eclíptico',
        aspectmode='cube',
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8))
    ),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
    width=800, height=650
)

fig_p.show()

---

## 8. Resumen de resultados

In [ ]:
# ============================================================
# Resumen cuantitativo del encuentro
# ============================================================

print("╔══════════════════════════════════════════════════════════╗")
print("║      RESUMEN: ENCUENTRO APOPHIS-TIERRA 2029              ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Tiempo del encuentro    : {2000+t_min:.4f} (≈ abril 2029)       ║")
print(f"║  Distancia mínima        : {d_min_km:>10.0f} km               ║")
print(f"║  En distancias lunares   : {d_min_dl:>10.4f} LD              ║")
print(f"║  Velocidad relativa      : {v_enc:>10.2f} km/s            ║")
print(f"║  Tiempo de cruce ~0.01UA : {(2*0.01*UA_KM/v_enc/3600):.1f} horas              ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Valor oficial JPL       :    38,017 km              ║")
print(f"║  Discrepancia simulación : {abs(d_min_km-38017):>10.0f} km               ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Parámetros orbitales de Apophis (J2000.0):              ║")
print(f"║    a = {elem_apophis['a']:.6f} UA    e = {elem_apophis['e']:.6f}                ║")
print(f"║    i = {np.degrees(elem_apophis['inc']):.4f}°     Ω = {np.degrees(elem_apophis['Omega']):.4f}°             ║")
print(f"║    ω = {np.degrees(elem_apophis['omega']):.4f}°     M = {np.degrees(elem_apophis['M']):.4f}°             ║")
print(f"║    T = {T_apophis:.4f} años  ({T_apophis*365.25:.1f} días)             ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Sistema de referencia: eclíptico heliocéntrico J2000.0  ║")
print("║  Integrador: IAS15 (Rein & Spiegel 2015)                 ║")
print(f"║  Error en energía (máx): {dE_rel.max():.1e}                    ║")
print("╚══════════════════════════════════════════════════════════╝")

---

## 9. Problemas propuestos

### 🟢 Nivel básico

**P1. Período orbital de Apophis**  
Modifica la simulación para integrar exactamente un período orbital de Apophis ($T_{\text{Apophis}} \approx 323.6$ días). Comprueba que Apophis regresa a su posición inicial con un error menor que $10^{-6}$ UA. ¿Cuántas vueltas da la Tierra mientras Apophis completa una?

**P2. Cambio del integrador**  
Reemplaza el integrador `ias15` por `whfast` y `leapfrog` para la simulación de 30 años. Compara el error relativo en la energía con cada uno. ¿Cuál conserva mejor la energía? ¿Por qué `ias15` es superior para encuentros cercanos?

**P3. Distancia en función de los elementos orbitales**  
Varía la anomalía media inicial de Apophis ($M_0$) en el rango $[180°, 220°]$ en pasos de $1°$. Para cada valor, calcula la distancia mínima de encuentro. Grafica $d_{\min}(M_0)$. ¿Existe algún valor de $M_0$ que produciría un impacto?

---

### 🟡 Nivel intermedio

**P4. Luna como perturbador**  
Agrega la Luna al sistema (masa $m_L = 3.69 \times 10^{-8}\,M_\odot$, semieje mayor respecto a la Tierra $a_L = 0.00257$ UA, $T_L = 27.3$ días). Compara la distancia mínima de Apophis con y sin la Luna. ¿Es significativo su efecto?

**P5. Esfera de Hill de la Tierra**  
El radio de la esfera de Hill de la Tierra es:
$$r_{\text{Hill}} = a_{\oplus} \left(\frac{m_{\oplus}}{3 M_{\odot}}\right)^{1/3}$$
Calcula cuánto tiempo permanece Apophis dentro de la esfera de Hill de la Tierra durante el encuentro de 2029. ¿Cambia significativamente su velocidad durante este tiempo?

**P6. Parámetro de impacto**  
El parámetro de impacto $b$ se define como la distancia de mínima aproximación que tendría Apophis si la Tierra no tuviera gravedad. Calcúlalo geométricamente a partir de las posiciones y velocidades relativas cuando Apophis entra en la esfera de Hill. Compara con la distancia de mínima aproximación real ¿por qué son diferentes?

---

### 🔴 Nivel avanzado

**P7. Efecto Yarkovsky (modelo simple)**  
Implementa una fuerza transversal (tangencial a la órbita) de magnitud $f_Y = 2.5 \times 10^{-15}\,\text{AU/day}^2$ usando la función `additional_forces` de REBOUND. Esta es una aproximación del efecto Yarkovsky detectado en Apophis. Integra 100 años y compara la distancia de encuentro de 2029 con y sin este efecto.

**P8. Keyholes y resonancias**  
Apophis podría entrar en una resonancia orbital 8:7 con la Tierra después del encuentro de 2029, lo que incrementaría las probabilidades de impacto en 2036. Simula la órbita de Apophis durante 10 años *después* del encuentro (2029–2039) y verifica si el sistema entra en resonancia: calcula el cociente de períodos $T_{\oplus}/T_{\text{Apophis}}$ a lo largo del tiempo.

**P9. Análisis de Monte Carlo**  
Los elementos orbitales de Apophis tienen incertidumbres. Realiza $N=500$ simulaciones donde la anomalía media inicial $M_0$ varía aleatoriamente con distribución normal $M_0 \sim \mathcal{N}(201.5°, 0.1°)$. Construye el histograma de la distancia mínima de encuentro. ¿Cuál es la probabilidad (empírica) de que la distancia sea menor de 30,000 km?

**P10. Perturbaciones de Júpiter**  
Agrega Júpiter al sistema ($m_J = 9.548 \times 10^{-4}\,M_\odot$, $a_J = 5.203$ UA) e integra durante 100 años. ¿Cómo varía la excentricidad de Apophis en ese período? Compara con el caso sin Júpiter y explica las variaciones seculares.

---

## 10. Referencias

### Software
- **Rein, H. & Liu, S.-F. (2012).** *REBOUND: An open-source multi-purpose N-body code for collisional dynamics.* A&A 537, A128. DOI: [10.1051/0004-6361/201116713](https://doi.org/10.1051/0004-6361/201116713)
- **Rein, H. & Spiegel, D.S. (2015).** *IAS15: a fast, adaptive, high-order integrator for gravitational dynamics.* MNRAS 446, 1424–1437. DOI: [10.1093/mnras/stu2164](https://doi.org/10.1093/mnras/stu2164)
- **Documentación REBOUND:** https://rebound.readthedocs.io/

### Datos orbitales
- **Farnocchia, D. et al. (2021).** *Apophis: The 2029 Earth Encounter.* The Planetary Science Journal 2, 2. DOI: [10.3847/PSJ/abc251](https://doi.org/10.3847/PSJ/abc251)
- **Brozović, M. et al. (2018).** *Goldstone and Arecibo radar observations of (99942) Apophis in 2012–2013.* Icarus 300, 115–128.
- **NASA JPL Horizons:** https://ssd.jpl.nasa.gov/horizons/
- **NASA CNEOS Close Approaches:** https://cneos.jpl.nasa.gov/ca/

### Contexto científico
- **Apophis source population and Earth encounter frequency of Apophis-like bodies** (arXiv: 2602.19849)
- **Proximity operations about Apophis through its 2029 Earth flyby** (arXiv: 2301.05315)
- **Prediction of Apophis's deformation-driven rotational evolution during its closest encounter to the Earth in 2029** (arXiv: 2401.00001)
- **Murray, C.D. & Dermott, S.F. (2000).** *Solar System Dynamics.* Cambridge University Press.

---

*Cuaderno elaborado para el curso de Mecánica Celeste — 2026*